# Módulo 06 · Series de tiempo

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Distinguir `Timestamp`, `Timedelta`, `Period` y `DateOffset`, y parsear fechas con control.
2. Construir series **regulares** a partir de eventos irregulares (`resample`, `asfreq`, `reindex`).
3. Agregar (*downsampling*) y desagregar (*upsampling*) con `resample`.
4. Calcular ventanas **móviles** (`rolling`), **acumuladas** (`expanding`) y **exponenciales** (`ewm`).
5. Comparar periodos con `shift`, `diff`, `pct_change` (crecimiento anual, YoY).
6. Manejar **zonas horarias** y horario de verano (DST).
7. Trabajar con **periodos** y **días hábiles**.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | Tipos de tiempo y parseo |
| 2 | De eventos a una serie regular |
| 3 | `resample` |
| 4 | `rolling` |
| 5 | `expanding` y `ewm` |
| 6 | `shift`, `diff`, `pct_change` y YoY |
| 7 | Zonas horarias |
| 8 | Periodos y días hábiles |
| 9 | Patrones día × hora |
| ✍️ | Ejercicios · ✅ Soluciones |

> 📌 La muestra tiene ~300 pedidos en 20 meses (≈ 0.5 pedidos al día). Las series diarias serán ruidosas y con muchos ceros: es un caso **muy realista** para practicar suavizado y agregación.

## 0 · Setup y carga

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

In [ ]:
FECHAS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]

orders = pd.read_csv(DATA / "olist_orders_dataset.csv",
                     dtype={"order_status": "category"}, parse_dates=FECHAS)

items = pd.read_csv(DATA / "olist_order_items_dataset.csv",
                    usecols=["order_id", "price", "freight_value"])

customers = pd.read_csv(DATA / "olist_customers_dataset.csv",
                        usecols=["customer_id", "customer_state"],
                        dtype={"customer_state": "category"})

# Una fila por pedido, con su ingreso (precio + flete) y la fecha de compra
ventas = (
  items.assign(ingreso=items["price"] + items["freight_value"])
       .groupby("order_id", as_index=False)["ingreso"].sum()
       .merge(orders[["order_id", "order_purchase_timestamp", "customer_id"]], on="order_id", validate="1:1")
       .merge(customers, on="customer_id", validate="m:1")
       .drop(columns="customer_id")
       .sort_values("order_purchase_timestamp")
       .reset_index(drop=True)
)
ventas.head()

## 1 · Tipos de tiempo y parseo

| Tipo | Qué representa | Ejemplo |
|---|---|---|
| `Timestamp` | un **instante** | `2018-03-15 14:30` |
| `Timedelta` | una **duración** exacta | `3 days 04:00:00` |
| `Period` | un **intervalo** con nombre | `2018-03` (todo marzo) |
| `DateOffset` | un salto **de calendario** | "fin de mes", "siguiente día hábil" |

In [ ]:
t = pd.Timestamp("2018-03-15 14:30")
print("Timestamp          :", t)
print("+ Timedelta 1 day  :", t + pd.Timedelta(days=1))
print("+ MonthEnd()       :", t + pd.offsets.MonthEnd())
print("+ BDay(3)          :", t + pd.offsets.BDay(3))
print("Period mensual     :", t.to_period("M"), "| empieza:", t.to_period("M").start_time.date(),
      "| termina:", t.to_period("M").end_time.date())

`Timedelta` es una duración **exacta**; un `DateOffset` respeta el calendario. Sumar "un mes" a 31 de enero no es lo mismo que sumar 30 días:

In [ ]:
ene31 = pd.Timestamp("2018-01-31")
print("+ 30 días        :", (ene31 + pd.Timedelta(days=30)).date())
print("+ DateOffset(1M) :", (ene31 + pd.DateOffset(months=1)).date())
print("+ MonthEnd()     :", (ene31 + pd.offsets.MonthEnd()).date())

### Parseo con control

- Declara el `format` cuando lo conoces: es más rápido y evita ambigüedades día/mes.
- `errors="coerce"` convierte lo imparseable en `NaT` en lugar de fallar.
- Observa la **resolución** del dtype: pandas 3 usa microsegundos (`[us]`) por defecto al parsear strings.

In [ ]:
crudas = pd.Series(["2018-03-15 14:30:00", "15/03/2018", "no es fecha", None])
print(pd.to_datetime(crudas, format="%Y-%m-%d %H:%M:%S", errors="coerce"))

In [ ]:
print(orders["order_purchase_timestamp"].dtype)

### Duraciones entre fechas

Restar fechas da un `Timedelta`. Para obtener un número: `.dt.days` (trunca), `.dt.total_seconds()` o dividir entre `pd.Timedelta(...)`.

In [ ]:
dur = orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
pd.DataFrame({
  "timedelta": dur,
  "dias_truncados": dur.dt.days,
  "dias_decimales": dur / pd.Timedelta(days=1),
}).head(3)

## 2 · De eventos a una serie regular

Los pedidos son **eventos irregulares**. Muchas operaciones (ventanas por número de filas, gráficos, modelos) necesitan una serie **regular**: un valor por día, incluidos los días sin pedidos.

In [ ]:
s = ventas.set_index("order_purchase_timestamp")["ingreso"]

# ❌ Agrupar por fecha deja fuera los días sin ventas
por_dia_incompleto = s.groupby(s.index.normalize()).sum()
print("Días con ventas         :", len(por_dia_incompleto))

# ✅ resample crea TODOS los días del rango
por_dia = s.resample("D").sum()
print("Días en el rango        :", len(por_dia))
print("Días con ingreso 0      :", (por_dia == 0).sum())

> ⚠️ `sum()` de un día vacío da **0**, pero `mean()` da **`NaN`**. Elige pensando en el significado: un día sin ventas tiene ingreso 0, pero su ticket promedio es indefinido.

In [ ]:
s.resample("D").agg(["sum", "mean", "size"]).loc["2017-01-28":"2017-02-05"]

### `asfreq` vs `resample` vs `reindex`

| Método | Qué hace con varios valores en un periodo | Qué hace con huecos |
|---|---|---|
| `resample(freq).agg(...)` | **agrega** | crea filas (0 o `NaN` según la función) |
| `asfreq(freq)` | toma el valor **en ese instante exacto** | `NaN` (o `method=`/`fill_value=`) |
| `reindex(date_range(...))` | exige índice único | `NaN` o `fill_value` |

`asfreq` sirve para series que **ya** son regulares, por ejemplo para pasar de mensual a diario:

In [ ]:
mensual = s.resample("ME").sum()
mensual.asfreq("D").head(3)                     # entre fines de mes sólo hay NaN

In [ ]:
# reindex para forzar un rango concreto (p. ej., todo 2017 aunque no haya datos al inicio)
anio_2017 = pd.date_range("2017-01-01", "2017-12-31", freq="D")
por_dia.reindex(anio_2017, fill_value=0).head(3)

## 3 · `resample`

`resample` es un **`groupby` por tiempo**: acepta `sum`, `mean`, `agg` con nombre, `apply`…

Frecuencias frecuentes en pandas 3 (los alias viejos `M`, `Q`, `Y` para fin de periodo se renombraron):

| Alias | Significado |
|---|---|
| `D`, `h`, `min` | día, hora, minuto |
| `W` / `W-MON` | semana que termina en domingo / en lunes |
| `ME` / `MS` | fin de mes / inicio de mes |
| `QE` / `QS` | fin / inicio de trimestre |
| `YE` / `YS` | fin / inicio de año |
| `B` | día hábil |

In [ ]:
semanal = ventas.resample("W", on="order_purchase_timestamp").agg(
  pedidos=("order_id", "size"),
  ingreso=("ingreso", "sum"),
  ticket=("ingreso", "mean"),
)
semanal.tail()

Observa `on=`: no hace falta poner la fecha en el índice. Y como cualquier `groupby`, puedes combinarlo con otras llaves usando `pd.Grouper` (Módulo 03):

In [ ]:
(ventas.groupby([pd.Grouper(key="order_purchase_timestamp", freq="QE"), "customer_state"], observed=True)
       ["ingreso"].sum()
       .unstack(fill_value=0)
       [["SP", "RJ", "MG"]])

### `label` y `closed`

Al agregar, ¿cada periodo se etiqueta con su **inicio** o su **fin**? Con `ME`/`W` el default es el **fin**; con `MS` el inicio. Para semanas puedes controlarlo:

In [ ]:
pd.DataFrame({
  "W (fin domingo)": s.resample("W").size().head(3).index,
  "W, label='left'": s.resample("W", label="left").size().head(3).index,
  "W-MON": s.resample("W-MON").size().head(3).index,
})

### Upsampling: de mensual a diario

In [ ]:
objetivo_mensual = pd.Series([3000.0, 3500.0, 4200.0],
                             index=pd.period_range("2018-01", periods=3, freq="M"))
diario = objetivo_mensual.resample("D").ffill()          # cada día hereda el objetivo de su mes
diario_prorrateado = diario / diario.index.days_in_month  # o repartido por día
diario_prorrateado.groupby(diario_prorrateado.index.month).sum()

## 4 · `rolling`

Una ventana **móvil** calcula una estadística sobre los últimos *n* puntos o el último *periodo* de tiempo.

| Ventana | Ejemplo | Cuándo |
|---|---|---|
| Por **número de filas** | `rolling(7)` | serie **regular** (una fila por día) |
| Por **tiempo** | `rolling("7D")` | eventos **irregulares** o con huecos |

Parámetros clave: `min_periods` (cuántos valores mínimos para dar resultado) y `center`.

In [ ]:
suave = pd.DataFrame({
  "diario": por_dia,
  "media_7d": por_dia.rolling(7).mean(),
  "media_28d": por_dia.rolling(28, min_periods=14).mean(),
})
ax = suave.loc["2017-06":].plot(figsize=(11, 4), alpha=0.8,
                                title="Ingreso diario y medias móviles")
ax.lines[0].set_alpha(0.25)

### Ventanas por tiempo sobre eventos irregulares

Con `rolling("7D")` sobre la serie **sin regularizar**, cada pedido mira los pedidos de los 7 días previos, sin importar cuántas filas sean. Es ideal para métricas "en los últimos N días" calculadas en el momento de cada evento.

In [ ]:
eventos = ventas.set_index("order_purchase_timestamp")
eventos = eventos.assign(
  pedidos_ult_7d=eventos["ingreso"].rolling("7D").count(),
  ingreso_ult_7d=eventos["ingreso"].rolling("7D").sum(),
)
eventos[["ingreso", "pedidos_ult_7d", "ingreso_ult_7d"]].loc["2018-02-01":"2018-02-05"]

> ⚠️ `rolling(7)` sobre eventos irregulares significaría "los últimos 7 **pedidos**", que pueden abarcar 2 días o 3 semanas. Asegúrate de que la ventana signifique lo que crees.

### Más estadísticas y ventanas por grupo

In [ ]:
por_dia.rolling(28).agg(["mean", "std", "max"]).dropna().tail(3)

In [ ]:
# Rolling por grupo: media móvil de 30 días del ticket por estado (eventos irregulares)
(ventas.set_index("order_purchase_timestamp")
       .groupby("customer_state", observed=True)["ingreso"]
       .rolling("30D").mean()
       .loc[["SP", "RJ"]]
       .groupby(level=0, observed=True).tail(2))

## 5 · `expanding` y `ewm`

- `expanding()`: ventana que **crece** desde el inicio (acumulados: media histórica, máximo histórico).
- `ewm(...)`: media móvil **exponencial**: pondera más lo reciente. Se define con `span`, `halflife` o `alpha`.

In [ ]:
mens = ventas.resample("ME", on="order_purchase_timestamp")["ingreso"].sum()

pd.DataFrame({
  "mensual": mens,
  "media_historica": mens.expanding().mean(),
  "max_historico": mens.expanding().max(),
  "ewm_span3": mens.ewm(span=3, adjust=False).mean(),
}).round(0).tail(6)

In [ ]:
# halflife con tiempo real: útil con eventos irregulares
eventos["ingreso"].ewm(halflife="14D", times=eventos.index).mean().tail(3)

## 6 · `shift`, `diff`, `pct_change` y YoY

| Método | Calcula |
|---|---|
| `shift(n)` | el valor de *n* periodos atrás (rezago) |
| `diff(n)` | `x - x.shift(n)` |
| `pct_change(n)` | `x / x.shift(n) - 1` |

Sobre una serie **regular**, `n` periodos tiene significado claro: con datos mensuales, `shift(12)` es "el mismo mes del año anterior".

In [ ]:
mens_p = mens.to_period("M")

comparativo = pd.DataFrame({
  "ingreso": mens_p,
  "mes_anterior": mens_p.shift(1),
  "var_mensual_%": mens_p.pct_change().mul(100),
  "mismo_mes_2017": mens_p.shift(12),
  "yoy_%": mens_p.pct_change(12).mul(100),
}).round(1)
comparativo.loc["2018"]

> 💡 `shift` también acepta `freq=`: `serie.shift(1, freq="D")` **mueve el índice** en lugar de los valores. Útil cuando la serie no es regular.

## 7 · Zonas horarias

Las fechas de Olist son **ingenuas** (*naive*): no dicen en qué zona horaria están. Sabemos que son hora de Brasil (São Paulo).

| Método | Hace |
|---|---|
| `tz_localize(zona)` | **declara** la zona de fechas ingenuas (no cambia la hora del reloj) |
| `tz_convert(zona)` | **convierte** a otra zona (cambia la hora del reloj, mismo instante) |
| `tz_localize(None)` | quita la zona |

In [ ]:
compra_sp = orders["order_purchase_timestamp"].dt.tz_localize("America/Sao_Paulo")
compra_mx = compra_sp.dt.tz_convert("America/Mexico_City")

pd.DataFrame({"ingenua": orders["order_purchase_timestamp"], "sao_paulo": compra_sp,
              "cdmx": compra_mx}).head(3)

### Horario de verano (DST)

Brasil tuvo horario de verano hasta 2019. En la noche del **15 de octubre de 2017** los relojes saltaron de 00:00 a 01:00 (esa hora **no existió**) y el **17 de febrero de 2018** a las 23:59 volvieron a 23:00 (esa hora **ocurrió dos veces**).

`tz_localize` lanza error en esos casos salvo que le digas qué hacer:

In [ ]:
casos = pd.Series(pd.to_datetime(["2017-10-15 00:30", "2018-02-17 23:30"]))

try:
  casos.dt.tz_localize("America/Sao_Paulo")
except Exception as e:
  print(type(e).__name__, "->", str(e)[:70])

In [ ]:
pd.DataFrame({
  "nonexistent=shift_forward": casos.iloc[[0]].dt.tz_localize("America/Sao_Paulo", nonexistent="shift_forward").values,
  "ambiguous=NaT": casos.iloc[[1]].dt.tz_localize("America/Sao_Paulo", ambiguous="NaT").values,
}, index=["valor"]).T

La diferencia horaria entre São Paulo y CDMX **no es constante** por el DST de Brasil (y el de México, vigente hasta 2022):

In [ ]:
dif_horas = (compra_sp.dt.tz_localize(None) - compra_mx.dt.tz_localize(None)) / pd.Timedelta(hours=1)
dif_horas.groupby(orders["order_purchase_timestamp"].dt.to_period("M").astype(str)).first().loc["2017-09":"2018-05"]

> 🎯 Buena práctica: guarda instantes en **UTC** (`tz_convert("UTC")`) y convierte a la zona local sólo para presentar o para análisis por hora del día.

## 8 · Periodos y días hábiles

### `Period` y `PeriodIndex`

Un `Period` es un intervalo con aritmética propia. Ideal para reportes mensuales/trimestrales: no hay ambigüedad de "¿el día 31 o el 1?".

In [ ]:
p = pd.Period("2018-02", freq="M")
print(p, "| días:", p.days_in_month, "| +1:", p + 1, "| trimestre:", p.asfreq("Q"))
print("Meses entre periodos:", (pd.Period("2018-08", "M") - pd.Period("2017-01", "M")).n)

In [ ]:
ventas.groupby(ventas["order_purchase_timestamp"].dt.to_period("Q"))["ingreso"].sum()

### Días hábiles

Para "días hábiles entre dos fechas", la herramienta más rápida y simple es `np.busday_count`, que acepta una lista de **feriados**. Trabaja con fechas de tipo `datetime64[D]`.

In [ ]:
FERIADOS_BR = pd.to_datetime([
  "2017-01-01", "2017-02-27", "2017-02-28", "2017-04-14", "2017-04-21", "2017-05-01",
  "2017-06-15", "2017-09-07", "2017-10-12", "2017-11-02", "2017-11-15", "2017-12-25",
  "2018-01-01", "2018-02-12", "2018-02-13", "2018-03-30", "2018-04-21", "2018-05-01",
  "2018-05-31", "2018-09-07", "2018-10-12", "2018-11-02", "2018-11-15", "2018-12-25",
])
feriados_np = FERIADOS_BR.values.astype("datetime64[D]")

entregados = orders.dropna(subset=["order_delivered_customer_date"])
inicio = entregados["order_purchase_timestamp"].values.astype("datetime64[D]")
fin = entregados["order_delivered_customer_date"].values.astype("datetime64[D]")

habiles = pd.Series(np.busday_count(inicio, fin, holidays=feriados_np), index=entregados.index)
naturales = (entregados["order_delivered_customer_date"].dt.normalize()
             - entregados["order_purchase_timestamp"].dt.normalize()).dt.days

pd.DataFrame({"naturales": naturales, "habiles": habiles}).describe().round(1)

> Los feriados listados son los **nacionales** de Brasil (incluye Carnaval y Corpus Christi, que son "puntos facultativos" muy observados). Revísalos si los usas en un análisis real.

Con *offsets* puedes **sumar** días hábiles:

In [ ]:
bday_br = pd.offsets.CustomBusinessDay(holidays=FERIADOS_BR)
pd.Timestamp("2018-02-09") + 3 * bday_br          # viernes antes de Carnaval

## 9 · Patrones día × hora

Una vista clásica: ¿cuándo compra la gente? Combinamos el accessor `.dt` con `pivot_table` (Módulo 04).

In [ ]:
DIAS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

patron = (
  orders.assign(dia=orders["order_purchase_timestamp"].dt.day_name(),
                franja=pd.cut(orders["order_purchase_timestamp"].dt.hour,
                              bins=[0, 6, 12, 18, 24], right=False,
                              labels=["00-06", "06-12", "12-18", "18-24"]))
        .pivot_table(index="dia", columns="franja", values="order_id",
                     aggfunc="count", fill_value=0, observed=False)
        .reindex(DIAS)
)
patron.style.background_gradient(cmap="Blues", axis=None)

---
# ✍️ Ejercicios

Variables disponibles: `orders`, `ventas` (un pedido por fila con `ingreso` y `customer_state`), `por_dia`, `FERIADOS_BR`, `feriados_np`.

### Ejercicio 1 · `resample` semanal ⭐

1. `sem`: a partir de `ventas`, DataFrame semanal con semanas que **terminan en domingo**, columnas `pedidos` (conteo) e `ingreso` (suma). Debe incluir las semanas sin pedidos.
2. `semanas_vacias`: número de semanas con 0 pedidos.
3. `mejor_semana`: la fecha (`Timestamp`) con la que está etiquetada la semana de mayor `ingreso`.

In [ ]:
# TU CÓDIGO AQUÍ
sem = ...
semanas_vacias = ...
mejor_semana = ...

In [ ]:
# Comprobación
if pendiente(sem, semanas_vacias, mejor_semana):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(sem.columns) == ["pedidos", "ingreso"]
  assert (sem.index.day_of_week == 6).all(), "Las semanas deben terminar en domingo"
  assert sem["pedidos"].sum() == len(ventas)
  assert np.isclose(sem["ingreso"].sum(), ventas["ingreso"].sum())
  assert semanas_vacias == 7, semanas_vacias
  assert mejor_semana == pd.Timestamp("2018-04-29"), mejor_semana
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · Ventanas móviles ⭐⭐

1. `ingreso_28`: media móvil de **28 días** de `por_dia` (serie diaria regular), con al menos 14 días de datos.
2. `pico_28`: la fecha donde `ingreso_28` alcanza su máximo.
3. `max_pedidos_7d`: usando `ventas` **sin regularizar**, el máximo de pedidos en cualquier ventana de 7 días terminando en un pedido (ventana por tiempo).
4. `iguales`: `True` si `por_dia.rolling(28).sum()` y `por_dia.rolling("28D").sum()` coinciden en todos los días donde ambos tienen valor (pista: piensa en `min_periods` por defecto de cada tipo de ventana y compara sólo donde ninguno es `NaN`).

In [ ]:
# TU CÓDIGO AQUÍ
ingreso_28 = ...
pico_28 = ...
max_pedidos_7d = ...
iguales = ...

In [ ]:
# Comprobación
if pendiente(ingreso_28, pico_28, max_pedidos_7d, iguales):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert len(ingreso_28) == len(por_dia)
  assert ingreso_28.iloc[:13].isna().all() and ingreso_28.iloc[13:].notna().all()
  assert pico_28 == pd.Timestamp("2018-05-10"), pico_28
  assert max_pedidos_7d == 13, max_pedidos_7d
  assert iguales is True or iguales == True
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Crecimiento anual (YoY) ⭐⭐

1. `mens_ing`: ingreso mensual de `ventas` como Series con `PeriodIndex` mensual, **de 2017-01 a 2018-08 completo** (meses sin ventas en 0).
2. `yoy`: crecimiento % de cada mes de 2018 contra el mismo mes de 2017 (Series indexada por los periodos de 2018).
3. `mes_mayor_yoy`: el periodo de 2018 con mayor crecimiento (como string `"2018-MM"`).
4. `yoy_acumulado`: crecimiento % del ingreso **acumulado** ene–ago 2018 vs ene–ago 2017.

In [ ]:
# TU CÓDIGO AQUÍ
mens_ing = ...
yoy = ...
mes_mayor_yoy = ...
yoy_acumulado = ...

In [ ]:
# Comprobación
if pendiente(mens_ing, yoy, mes_mayor_yoy, yoy_acumulado):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(mens_ing.index, pd.PeriodIndex) and len(mens_ing) == 20
  assert str(mens_ing.index[0]) == "2017-01" and str(mens_ing.index[-1]) == "2018-08"
  assert len(yoy) == 8 and all(str(p).startswith("2018") for p in yoy.index)
  assert mes_mayor_yoy == "2018-01", mes_mayor_yoy
  assert np.isclose(yoy_acumulado, 280.859, atol=0.05), yoy_acumulado
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · Zonas horarias ⭐⭐

Un equipo en CDMX quiere saber cuántas compras de Brasil ocurrieron **fuera de su horario laboral** (antes de las 9:00 o desde las 18:00, hora de CDMX, o en sábado/domingo de CDMX).

1. `compra_cdmx`: `order_purchase_timestamp` localizada en `"America/Sao_Paulo"` (ambiguas y no existentes → `NaT`) y convertida a `"America/Mexico_City"`.
2. `pct_fuera_cdmx`: % de compras fuera del horario laboral de CDMX.
3. `pct_fuera_sp`: el mismo cálculo pero con la hora local de **São Paulo**.
4. `n_cambia_dia`: número de compras cuyo **día calendario** es distinto en CDMX que en São Paulo.

In [ ]:
# TU CÓDIGO AQUÍ
compra_cdmx = ...
pct_fuera_cdmx = ...
pct_fuera_sp = ...
n_cambia_dia = ...

In [ ]:
# Comprobación
if pendiente(compra_cdmx, pct_fuera_cdmx, pct_fuera_sp, n_cambia_dia):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert str(compra_cdmx.dt.tz) == "America/Mexico_City"
  assert len(compra_cdmx) == len(orders)
  assert np.isclose(pct_fuera_cdmx, 54.6667, atol=0.01), pct_fuera_cdmx
  assert np.isclose(pct_fuera_sp, 55.6667, atol=0.01), pct_fuera_sp
  assert n_cambia_dia == 10, n_cambia_dia
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: promesa de entrega en días hábiles ⭐⭐⭐

Para pedidos **entregados** (con fecha de entrega):

1. `hab`: DataFrame con `order_id`, `habiles_real` (días hábiles de compra a entrega) y `habiles_prometido` (días hábiles de compra a fecha estimada), usando `np.busday_count` y los feriados de Brasil.
2. `tarde_hab`: Series booleana: `habiles_real > habiles_prometido`.
3. `tarde_cal`: Series booleana: la fecha de entrega (**día calendario**, sin hora) es posterior a la fecha estimada.
4. `n_discrepan`: pedidos donde `tarde_hab` y `tarde_cal` no coinciden. Si en la muestra sale 0, explica en un comentario **en qué caso** podrían diferir.
5. `mediana_hab_por_mes`: mediana de `habiles_real` por **mes de compra** (`PeriodIndex`).

In [ ]:
# TU CÓDIGO AQUÍ
hab = ...
tarde_hab = ...
tarde_cal = ...
n_discrepan = ...
mediana_hab_por_mes = ...

In [ ]:
# Comprobación
if pendiente(hab, tarde_hab, tarde_cal, n_discrepan, mediana_hab_por_mes):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(hab.columns) == ["order_id", "habiles_real", "habiles_prometido"]
  assert len(hab) == orders["order_delivered_customer_date"].notna().sum()
  assert tarde_hab.sum() == 19, tarde_hab.sum()
  assert tarde_cal.sum() == 19, tarde_cal.sum()
  assert n_discrepan == 0, n_discrepan
  assert isinstance(mediana_hab_por_mes.index, pd.PeriodIndex)
  assert np.isclose(mediana_hab_por_mes.loc["2018-02"], 6.5), mediana_hab_por_mes.loc["2018-02"]
  print("✅ Ejercicio 5 correcto")

---
# ✅ Soluciones

### Solución 1

In [ ]:
sem = ventas.resample("W", on="order_purchase_timestamp").agg(
  pedidos=("order_id", "size"),
  ingreso=("ingreso", "sum"),
)
semanas_vacias = (sem["pedidos"] == 0).sum()
mejor_semana = sem["ingreso"].idxmax()
semanas_vacias, mejor_semana

*Nota:* `"W"` equivale a `"W-SUN"`: cada semana se etiqueta con el **domingo** en que termina.

### Solución 2

In [ ]:
ingreso_28 = por_dia.rolling(28, min_periods=14).mean()
pico_28 = ingreso_28.idxmax()

max_pedidos_7d = int(ventas.set_index("order_purchase_timestamp")["ingreso"].rolling("7D").count().max())

r_filas = por_dia.rolling(28).sum()       # min_periods = 28 por defecto
r_tiempo = por_dia.rolling("28D").sum()   # min_periods = 1 por defecto
ambos = r_filas.notna() & r_tiempo.notna()
iguales = bool(np.allclose(r_filas[ambos], r_tiempo[ambos]))
pico_28, max_pedidos_7d, iguales

*Nota:* en una serie diaria **completa**, 28 filas = 28 días, por eso coinciden. La diferencia está en `min_periods`: las ventanas por número de filas exigen la ventana completa por defecto; las ventanas por tiempo devuelven resultado desde el primer dato. Si la serie tuviera huecos (días faltantes), los valores también diferirían.

### Solución 3

In [ ]:
mens_ing = (
  ventas.groupby(ventas["order_purchase_timestamp"].dt.to_period("M"))["ingreso"].sum()
        .reindex(pd.period_range("2017-01", "2018-08", freq="M"), fill_value=0)
)

yoy = mens_ing.pct_change(12).mul(100).loc["2018"]
mes_mayor_yoy = str(yoy.idxmax())

ac_2018 = mens_ing.loc["2018-01":"2018-08"].sum()
ac_2017 = mens_ing.loc["2017-01":"2017-08"].sum()
yoy_acumulado = (ac_2018 / ac_2017 - 1) * 100
yoy.round(1), mes_mayor_yoy, round(yoy_acumulado, 2)

*Nota:* el `reindex` con `period_range` garantiza que `pct_change(12)` compare **exactamente** el mismo mes del año anterior; si faltara un mes, `shift(12)` se desfasaría. El YoY de enero es enorme porque enero de 2017 casi no tiene ventas en la muestra: con bases pequeñas, los porcentajes engañan.

### Solución 4

In [ ]:
compra_sp_tz = orders["order_purchase_timestamp"].dt.tz_localize(
  "America/Sao_Paulo", ambiguous="NaT", nonexistent="NaT")
compra_cdmx = compra_sp_tz.dt.tz_convert("America/Mexico_City")


def fuera_de_horario(ts: pd.Series) -> pd.Series:
  return (ts.dt.hour < 9) | (ts.dt.hour >= 18) | (ts.dt.day_of_week >= 5)


pct_fuera_cdmx = fuera_de_horario(compra_cdmx).mean() * 100
pct_fuera_sp = fuera_de_horario(compra_sp_tz).mean() * 100
n_cambia_dia = (compra_cdmx.dt.date != compra_sp_tz.dt.date).sum()
round(pct_fuera_cdmx, 2), round(pct_fuera_sp, 2), n_cambia_dia

*Nota:* el mismo instante cae en distinta hora (e incluso distinto **día**) según la zona. São Paulo va **adelante** de CDMX (entre 2 y 4 horas según el DST de cada país), así que una compra hecha de madrugada en Brasil todavía pertenece al **día anterior** en CDMX. Por eso `n_cambia_dia` no es cero.

### Solución 5

In [ ]:
ent = orders.dropna(subset=["order_delivered_customer_date"])

a_dia = lambda s: s.values.astype("datetime64[D]")
compra_d = a_dia(ent["order_purchase_timestamp"])

hab = pd.DataFrame({
  "order_id": ent["order_id"].values,
  "habiles_real": np.busday_count(compra_d, a_dia(ent["order_delivered_customer_date"]), holidays=feriados_np),
  "habiles_prometido": np.busday_count(compra_d, a_dia(ent["order_estimated_delivery_date"]), holidays=feriados_np),
})

tarde_hab = hab["habiles_real"] > hab["habiles_prometido"]
tarde_cal = pd.Series(
  ent["order_delivered_customer_date"].dt.normalize().values > ent["order_estimated_delivery_date"].dt.normalize().values
)
n_discrepan = (tarde_hab.values != tarde_cal.values).sum()

mediana_hab_por_mes = (
  hab.assign(mes=ent["order_purchase_timestamp"].dt.to_period("M").array)
     .groupby("mes")["habiles_real"].median()
)

# Respuesta 4: np.busday_count cuenta días hábiles en [inicio, fin). Si la promesa cae
# en un día NO hábil (sábado, domingo o feriado) y la entrega ocurre después pero antes del
# siguiente día hábil, ambos conteos son iguales: p. ej., promesa el sábado y entrega el
# domingo es "tarde" en calendario pero no en días hábiles. En esta muestra no ocurre.
tarde_hab.sum(), tarde_cal.sum(), n_discrepan, mediana_hab_por_mes.tail(4)

---
## Resumen del módulo

- `Timedelta` = duración exacta; `DateOffset` = salto de calendario; `Period` = intervalo.
- Regulariza eventos con `resample` (o `reindex` sobre un `date_range`/`period_range`) antes de usar ventanas por número de filas o `shift`.
- `sum` de un periodo vacío es 0; `mean` es `NaN`.
- Alias pandas 3: `ME`, `QE`, `YE` (fin), `MS`, `QS`, `YS` (inicio).
- `rolling(n)` = últimas *n* filas; `rolling("7D")` = últimos 7 días (ideal para eventos irregulares). Cuida `min_periods`.
- `expanding` para acumulados; `ewm` para suavizado que pondera lo reciente.
- YoY: serie mensual completa + `pct_change(12)`.
- `tz_localize` declara, `tz_convert` convierte; maneja DST con `ambiguous`/`nonexistent`.
- Días hábiles: `np.busday_count` con feriados; sumar con `CustomBusinessDay`.

**Siguiente:** Módulo 07 · Texto y datos sucios: el accessor `.str`, regex y validación.